# add-sub-div-back-lambdas — faded example 3: Complete grad_s dispatch in reverse pass of z = (p - q) / r

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `add-sub-div-back-lambdas`. Running the beacon reports progress on the `Backprop: add/sub/div back as lambdas` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: add/sub/div back as lambdas` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`add-sub-div-back-lambdas`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "add-sub-div-back-lambdas"
DD_SUBTOPIC = "Backprop: add/sub/div back as lambdas"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Walking the reverse pass of `s = p - q; z = s / r`, the div op's arg0 backward produces `grad_s` (the gradient w.r.t. the div's first input `s`), which is then fed into the sub backwards to split to `p` and `q`. The div arg0 lambda returns `grad_z / r`, so `grad_s = grad_z / r`.

## Faded exercise 3

### Finish the reverse pass

`mini_back` and most of `reverse_sub_div_chain(p, q, r)` for `z = (p - q) / r` are given. Complete the single dispatch that computes `grad_s` (gradient flowing back into `s = p - q`) by routing `grad_z` through the div arg0 backward.

**Fill in:** grad_s computed by dispatching div argnum 0 on grad_z, which equals grad_z / r

In [ ]:
BACK = {
    ('sub', 0): lambda g, o, x, y: g,
    ('sub', 1): lambda g, o, x, y: -g,
    ('div', 0): lambda g, o, x, y: g / y,
    ('div', 1): lambda g, o, x, y: -g * x / (y * y),
}

def mini_back(op_name, argnum, grad_out, out, x, y):
    return BACK[(op_name, argnum)](grad_out, out, x, y)

def reverse_sub_div_chain(p, q, r):
    s = p - q
    z = s / r
    grad_z = t.ones_like(z)
    grad_s = None  # TODO: grad_s computed by dispatching div argnum 0 on grad_z, which equals grad_z / r
    grad_r = mini_back('div', 1, grad_z, z, s, r)
    grad_p = mini_back('sub', 0, grad_s, s, p, q)
    grad_q = mini_back('sub', 1, grad_s, s, p, q)
    return {'dp': grad_p, 'dq': grad_q, 'dr': grad_r}


def _test():
    t.manual_seed(3)
    p = t.randn(5, requires_grad=True)
    q = t.randn(5, requires_grad=True)
    r = (t.randn(5).abs() + 0.5).requires_grad_(True)
    out = (p - q) / r
    out.sum().backward()
    grads = reverse_sub_div_chain(p.detach(), q.detach(), r.detach())
    assert t.allclose(grads['dp'], p.grad, atol=1e-5)
    assert t.allclose(grads['dq'], q.grad, atol=1e-5)
    assert t.allclose(grads['dr'], r.grad, atol=1e-5)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
BACK = {
    ('sub', 0): lambda g, o, x, y: g,
    ('sub', 1): lambda g, o, x, y: -g,
    ('div', 0): lambda g, o, x, y: g / y,
    ('div', 1): lambda g, o, x, y: -g * x / (y * y),
}

def mini_back(op_name, argnum, grad_out, out, x, y):
    return BACK[(op_name, argnum)](grad_out, out, x, y)

def reverse_sub_div_chain(p, q, r):
    s = p - q
    z = s / r
    grad_z = t.ones_like(z)
    grad_s = mini_back('div', 0, grad_z, z, s, r)
    grad_r = mini_back('div', 1, grad_z, z, s, r)
    grad_p = mini_back('sub', 0, grad_s, s, p, q)
    grad_q = mini_back('sub', 1, grad_s, s, p, q)
    return {'dp': grad_p, 'dq': grad_q, 'dr': grad_r}
```
</details>